In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Raw:", RAW_DIR)
print("Interim:", INTERIM_DIR)

Raw: /Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/raw
Interim: /Users/kartiksinghai/Desktop/nse-stock-reversal-predictor/data/interim


In [2]:
tickers_df = pd.read_csv(RAW_DIR / "tickers.csv")

all_stocks = []

for ticker in tickers_df["ticker"]:
    filename = ticker.replace(".NS", "") + ".csv"
    filepath = RAW_DIR / filename

    df = pd.read_csv(filepath, parse_dates=["Date"])
    all_stocks.append(df)

stocks = pd.concat(all_stocks, ignore_index=True)

# Keep a clean, consistent ordering
stocks = stocks.sort_values(["ticker", "Date"]).reset_index(drop=True)

print("Combined shape:", stocks.shape)
stocks.head()

Combined shape: (121775, 10)


,Date,Adj Close,Close,High,Low,Open,Volume,ticker,company,sector
0,2015-01-01,300.506439,319.549988,322.500000,316.250000,319.000000,1456204,ADANIPORTS.NS,Adani Ports,Industrials
1,2015-01-02,300.318268,319.350006,325.799988,318.049988,319.350006,2894058,ADANIPORTS.NS,Adani Ports,Industrials
2,2015-01-05,304.503113,323.799988,327.500000,319.350006,320.450012,2099786,ADANIPORTS.NS,Adani Ports,Industrials
3,2015-01-06,302.669373,321.850006,331.450012,315.600006,321.649994,3672197,ADANIPORTS.NS,Adani Ports,Industrials
4,2015-01-07,301.964050,321.100006,328.700012,317.399994,321.950012,2981544,ADANIPORTS.NS,Adani Ports,Industrials


In [3]:
required_columns = [
    "Date", "Adj Close", "Close", "High",
    "Low", "Open", "Volume",
    "ticker", "company", "sector"
]

quality_report = {
    "total_rows": len(stocks),
    "unique_stocks": stocks["ticker"].nunique(),
    "earliest_date": stocks["Date"].min(),
    "latest_date": stocks["Date"].max(),
    "duplicate_ticker_date_rows": stocks.duplicated(
        subset=["ticker", "Date"]
    ).sum(),
    "missing_required_values": stocks[required_columns].isna().sum().sum(),
    "zero_or_negative_adj_close": (stocks["Adj Close"] <= 0).sum(),
    "negative_volume": (stocks["Volume"] < 0).sum(),
}

pd.Series(quality_report)

total_rows                                 121775
unique_stocks                                  43
earliest_date                 2015-01-01 00:00:00
latest_date                   2026-06-19 00:00:00
duplicate_ticker_date_rows                      0
missing_required_values                         0
zero_or_negative_adj_close                      0
negative_volume                                 0
dtype: object

In [4]:
stock_summary = (
    stocks.groupby("ticker")
    .agg(
        rows=("Date", "size"),
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        missing_adj_close=("Adj Close", lambda x: x.isna().sum()),
        zero_volume_days=("Volume", lambda x: (x == 0).sum())
    )
    .sort_values("rows")
)

stock_summary.head(10)

,rows,start_date,end_date,missing_adj_close,zero_volume_days
ticker,,,,,
DIVISLAB.NS,2831,2015-01-01,2026-06-19,0,3
ADANIPORTS.NS,2832,2015-01-01,2026-06-19,0,5
JSWSTEEL.NS,2832,2015-01-01,2026-06-19,0,9
KOTAKBANK.NS,2832,2015-01-01,2026-06-19,0,4
LT.NS,2832,2015-01-01,2026-06-19,0,4
M&M.NS,2832,2015-01-01,2026-06-19,0,5
MARUTI.NS,2832,2015-01-01,2026-06-19,0,5
NESTLEIND.NS,2832,2015-01-01,2026-06-19,0,4
NTPC.NS,2832,2015-01-01,2026-06-19,0,4


In [5]:
gap_check = stocks[["ticker", "Date"]].copy()
gap_check["previous_date"] = gap_check.groupby("ticker")["Date"].shift(1)
gap_check["gap_days"] = (
    gap_check["Date"] - gap_check["previous_date"]
).dt.days

large_gaps = gap_check[gap_check["gap_days"] > 10].sort_values(
    ["ticker", "gap_days"],
    ascending=[True, False]
)

large_gaps.head(20)

,ticker,Date,previous_date,gap_days


In [6]:
stocks.to_parquet(
    INTERIM_DIR / "combined_nse_ohlcv_raw.parquet",
    index=False
)

stock_summary.to_csv(
    INTERIM_DIR / "stock_level_quality_summary.csv"
)

print("Saved combined dataset and quality summary.")

Saved combined dataset and quality summary.
